In [1]:
import os, zipfile, glob

for r, d, files in os.walk("/kaggle/input"):
    for f in files:
        p = os.path.join(r, f)
        print(p, round(os.path.getsize(p)/1e6, 1), "MB")

z = glob.glob("/kaggle/input/**/MicroLens_1M_x2.zip", recursive=True)
if z and not glob.glob("/kaggle/input/**/train.parquet", recursive=True):
    os.makedirs("/kaggle/working/x2", exist_ok=True)
    zipfile.ZipFile(z[0]).extractall("/kaggle/working/x2")
    os.system("ls -lh /kaggle/working/x2")

/kaggle/input/datasets/salmabrika/microlens-x2/README 0.0 MB
/kaggle/input/datasets/salmabrika/microlens-x2/item_info.parquet 52.9 MB
/kaggle/input/datasets/salmabrika/microlens-x2/item_raw_feature.parquet 619.4 MB
/kaggle/input/datasets/salmabrika/microlens-x2/test_unlabeled.parquet 45.1 MB
/kaggle/input/datasets/salmabrika/microlens-x2/train.parquet 252.3 MB
/kaggle/input/datasets/salmabrika/microlens-x2/valid.parquet 43.9 MB


In [2]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score
D = "/kaggle/input/datasets/salmabrika/microlens-x2/"

train = pd.read_parquet(D+"train.parquet")
valid = pd.read_parquet(D+"valid.parquet")
test  = pd.read_parquet(D+"test_unlabeled.parquet")
info  = pd.read_parquet(D+"item_info.parquet")

for n, df in [("train",train),("valid",valid),("test",test),("info",info)]:
    print(n, df.shape, list(df.columns))

print("\ntest (attendu 343025):", len(test))
print("label moyen train / valid:", train["label"].mean(), valid["label"].mean())
print(train.head(2).T)

for c in ["likes_level","views_level"]:
    if c in valid.columns:
        print("AUC", c, "seul:", roc_auc_score(valid["label"], valid[c]))

print("\nmax item_id train/valid/test:", train.item_id.max(), valid.item_id.max(), test.item_id.max())
tr = set(train.item_id)
print("items de valid vus dans train:", valid.item_id.isin(tr).mean())
print("items de test vus dans train:", test.item_id.isin(tr).mean())
print("longueur historique:", train["item_seq"].map(len).describe())

train (6155540, 8) ['user_id', 'item_id', 'timestamp', 'hist_seq', 'hist_seq_ts', 'label', 'likes_level', 'views_level']
valid (340220, 8) ['user_id', 'item_id', 'timestamp', 'hist_seq', 'hist_seq_ts', 'label', 'likes_level', 'views_level']
test (343025, 8) ['user_id', 'item_id', 'timestamp', 'hist_seq', 'hist_seq_ts', 'label', 'likes_level', 'views_level']
info (91718, 3) ['item_id', 'item_tags', 'item_emb_d128']

test (attendu 343025): 343025
label moyen train / valid: 0.2 0.2
                                                             0  \
user_id                                                      1   
item_id                                                   8022   
timestamp                                        1661577914663   
hist_seq     [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
hist_seq_ts  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...   
label                                                        1   
likes_level                                           

KeyError: 'item_seq'

In [3]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

s = train.sample(200_000, random_state=0)
L = s["hist_seq"].map(lambda x: int((np.asarray(x) != 0).sum()))
print("taille du tableau hist_seq:", len(train["hist_seq"].iloc[0]))
print(L.describe())
print("part d'historiques vides:", (L == 0).mean())

for n, df in [("train", train), ("valid", valid), ("test", test)]:
    print(n, "timestamps:", pd.to_datetime(df.timestamp.min(), unit="ms"),
          "->", pd.to_datetime(df.timestamp.max(), unit="ms"))

print("labels du test:\n", test["label"].value_counts(dropna=False).head())
tu = set(train.user_id)
print("users de valid vus dans train:", valid.user_id.isin(tu).mean(),
      "| test:", test.user_id.isin(tu).mean())

g = train.groupby(["user_id", "timestamp"]).size()
print("lignes par (user, timestamp):\n", g.describe())

inh = np.fromiter((i in h for i, h in zip(s["item_id"], s["hist_seq"])), dtype=bool)
print("cible dans l'historique par label:",
      pd.Series(inh).groupby(s["label"].values).mean().to_dict())
print("AUC 'cible dans l'historique':", roc_auc_score(s["label"], inh))

ctr = train.groupby("item_id")["label"].mean()
print("AUC CTR-par-item (train -> valid):",
      roc_auc_score(valid["label"], valid["item_id"].map(ctr).fillna(0.2)))
print("item_info: 1re ligne item_id =", info.iloc[0]["item_id"], "| max =", info.item_id.max())

taille du tableau hist_seq: 50
count    200000.000000
mean         11.228150
std          10.345273
min           4.000000
25%           5.000000
50%           8.000000
75%          12.000000
max          50.000000
Name: hist_seq, dtype: float64
part d'historiques vides: 0.0
train timestamps: 2021-11-03 21:46:34.295000 -> 2022-09-12 12:02:45.275000
valid timestamps: 2022-03-04 01:36:05.205000 -> 2022-09-12 12:02:38.035000
test timestamps: 2022-03-27 12:11:33.141000 -> 2022-09-12 12:01:56.800000
labels du test:
 label
NaN    343025
Name: count, dtype: int64
users de valid vus dans train: 0.0 | test: 0.0
lignes par (user, timestamp):
 count    1231108.0
mean           5.0
std            0.0
min            5.0
25%            5.0
50%            5.0
75%            5.0
max            5.0
dtype: float64
cible dans l'historique par label: {0: 0.0, 1: 0.0}
AUC 'cible dans l'historique': 0.5
AUC CTR-par-item (train -> valid): 0.6977245510902995
item_info: 1re ligne item_id = 0 | max = 91717


In [4]:
import os, zipfile
g = train.groupby(["user_id","timestamp"])["label"].agg(["size","sum"])
print("positifs par groupe de 5:\n", g["sum"].value_counts())

k = train[["user_id","timestamp"]].values
print("train: groupes contigus par 5 ?", (k[::5] == k[4::5]).all())
pos = (np.arange(len(train)) % 5)[train["label"].values == 1]
print("position du positif dans le groupe:\n", pd.Series(pos).value_counts(normalize=True).sort_index())

kt = test[["user_id","timestamp"]].values
print("test: groupes contigus par 5 ?", (kt[::5] == kt[4::5]).all(), "| nb de groupes:", len(test)//5)

positifs par groupe de 5:
 sum
1    1231108
Name: count, dtype: int64
train: groupes contigus par 5 ? True
position du positif dans le groupe:
 0    1.0
Name: proportion, dtype: float64
test: groupes contigus par 5 ? False | nb de groupes: 68605


In [5]:
ctr = train.groupby("item_id")["label"].mean()
prior = train["label"].mean()
print("AUC valid:", roc_auc_score(valid["label"], valid["item_id"].map(ctr).fillna(prior)))

pred = test["item_id"].map(ctr).fillna(prior).values
sub = pd.DataFrame({"ID": np.arange(len(test)), "Task1&2": pred})
assert len(sub) == 343025 and sub["Task1&2"].between(0, 1).all() and not sub.isna().any().any()

os.makedirs("/kaggle/working/sub", exist_ok=True)
sub.to_csv("/kaggle/working/sub/prediction.csv", index=False)
with zipfile.ZipFile("/kaggle/working/sub/prediction.zip", "w", zipfile.ZIP_DEFLATED) as z:
    z.write("/kaggle/working/sub/prediction.csv", arcname="prediction.csv")
print(open("/kaggle/working/sub/prediction.csv").read()[:150])

AUC valid: 0.6977245510902995
ID,Task1&2
0,0.11607142857142858
1,0.15204678362573099
2,0.2455573505654281
3,0.16459627329192547
4,0.1258741258741259
5,0.3875
6,0.15853658536585366



In [6]:
from IPython.display import FileLink
FileLink("/kaggle/working/sub/prediction.zip")

/kaggle/working/sub/prediction.zip

In [7]:
print("valid: groupes contigus par 5 ?",
      (valid[["user_id","timestamp"]].values[::5] == valid[["user_id","timestamp"]].values[4::5]).all())
print("valid: position du positif:\n",
      pd.Series((np.arange(len(valid)) % 5)[valid["label"].values == 1]).value_counts(normalize=True).sort_index())
print("test: taille des groupes (user, timestamp):\n",
      test.groupby(["user_id","timestamp"]).size().value_counts())

valid: groupes contigus par 5 ? False
valid: position du positif:
 0    0.200341
1    0.199753
2    0.199004
3    0.199915
4    0.200988
Name: proportion, dtype: float64
test: taille des groupes (user, timestamp):
 5    68605
Name: count, dtype: int64


In [1]:
import os, gc, time, zipfile, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score
D = "/kaggle/input/datasets/salmabrika/microlens-x2/"
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu"); print("device:", dev)

cols = ["item_id", "hist_seq", "likes_level", "views_level"]
def prep(name, label=True):
    df = pd.read_parquet(D + name, columns=cols + (["label"] if label else []))
    S = dict(h=torch.from_numpy(np.stack(df["hist_seq"].values).astype(np.int32)).to(dev),
             t=torch.tensor(df["item_id"].values, dtype=torch.int32, device=dev),
             l=torch.tensor(df["likes_level"].values, dtype=torch.int32, device=dev),
             v=torch.tensor(df["views_level"].values, dtype=torch.int32, device=dev))
    if label:
        S["y"] = torch.tensor(df["label"].values, dtype=torch.float32, device=dev)
    del df; gc.collect()
    return S

TR = prep("train.parquet"); VA = prep("valid.parquet"); TE = prep("test_unlabeled.parquet", label=False)
NL = int(max(TR["l"].max(), VA["l"].max(), TE["l"].max())) + 1
NV = int(max(TR["v"].max(), VA["v"].max(), TE["v"].max())) + 1
info = pd.read_parquet(D + "item_info.parquet")
print({k: tuple(v.shape) for k, v in TR.items()}, "NL/NV:", NL, NV)

device: cuda
{'h': (6155540, 50), 't': (6155540,), 'l': (6155540,), 'v': (6155540,), 'y': (6155540,)} NL/NV: 11 11


In [3]:
VOCAB = int(info.item_id.max()) + 1
E_org = np.zeros((VOCAB, 128), dtype=np.float32)
E_org[info.item_id.values] = np.stack(info["item_emb_d128"].values).astype(np.float32)
print(E_org.shape, "ligne 0 nulle:", not E_org[0].any())

(91718, 128) ligne 0 nulle: True


In [5]:
class DIN(nn.Module):
    def __init__(self, E, nl, nv, d=128, drop=0.3):
        super().__init__()
        self.emb = nn.Embedding.from_pretrained(torch.tensor(E, dtype=torch.float32), freeze=False, padding_idx=0)
        self.le, self.ve = nn.Embedding(nl, 16), nn.Embedding(nv, 16)
        self.att = nn.Sequential(nn.Linear(4*d, 128), nn.PReLU(), nn.Linear(128, 64), nn.PReLU(), nn.Linear(64, 1))
        self.mlp = nn.Sequential(nn.Linear(3*d + 32, 256), nn.ReLU(), nn.Dropout(drop),
                                 nn.Linear(256, 128), nn.ReLU(), nn.Dropout(drop), nn.Linear(128, 1))
    def forward(self, h, t, l, v):
        te, he = self.emb(t), self.emb(h)
        tx = te.unsqueeze(1).expand_as(he)
        a = self.att(torch.cat([tx, he, tx - he, tx * he], -1)).squeeze(-1)
        a = a.masked_fill(h == 0, -1e4)
        w = torch.softmax(a.float(), 1).to(he.dtype)
        it = (w.unsqueeze(-1) * he).sum(1)
        x = torch.cat([te, it, te * it, self.le(l), self.ve(v)], -1)
        return self.mlp(x).squeeze(-1)

def fwd(m, S, idx):
    with torch.autocast(dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
        return m(S["h"][idx].long(), S["t"][idx].long(), S["l"][idx].long(), S["v"][idx].long())

@torch.no_grad()
def predict(m, S, bs=8192):
    m.eval(); out = []
    n = len(S["t"])
    for i in range(0, n, bs):
        idx = torch.arange(i, min(i + bs, n), device=dev)
        out.append(torch.sigmoid(fwd(m, S, idx).float()).cpu())
    return torch.cat(out).numpy()

def valid_auc(m):
    return roc_auc_score(VA["y"].cpu().numpy(), predict(m, VA))

def train(E, epochs=4, bs=2048, lr=1e-3, wd=1e-5, drop=0.3, seed=0):
    torch.manual_seed(seed)
    m = DIN(E, NL, NV, drop=drop).to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
    n = len(TR["t"]); best_auc, best_state = 0, None
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n, device=dev); t0 = time.time(); tot = torch.zeros((), device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(fwd(m, TR, idx).float(), TR["y"][idx])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.detach()
        auc = valid_auc(m)
        print(f"epoch {ep+1}: loss={tot.item()/(n/bs):.4f}  valid AUC={auc:.4f}  ({time.time()-t0:.0f}s)")
        if auc > best_auc:
            best_auc = auc; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    return m, best_auc

In [6]:
model_org, auc_org = train(E_org, epochs=4)
print("meilleure AUC valid (embeddings organisateurs):", auc_org)
torch.save(model_org.state_dict(), "/kaggle/working/model_org.pt")

epoch 1: loss=0.4525  valid AUC=0.7372  (77s)
epoch 2: loss=0.4184  valid AUC=0.7361  (76s)
epoch 3: loss=0.3610  valid AUC=0.6986  (76s)
epoch 4: loss=0.2787  valid AUC=0.6628  (76s)
meilleure AUC valid (embeddings organisateurs): 0.7372290426596665


In [10]:
def make_submission(m, tag="prediction"):
    p = predict(m, TE)
    sub = pd.DataFrame({"ID": np.arange(len(p)), "Task1&2": p})
    assert len(sub) == 343025 and sub["Task1&2"].between(0, 1).all() and not sub.isna().any().any()
    os.makedirs("/kaggle/working/sub", exist_ok=True)
    sub.to_csv("/kaggle/working/sub/prediction.csv", index=False)
    with zipfile.ZipFile(f"/kaggle/working/sub/{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("/kaggle/working/sub/prediction.csv", arcname="prediction.csv")
    print(sub.head(3)); print(sub["Task1&2"].describe())

make_submission(model_org, "prediction_din_org")
from IPython.display import FileLink
FileLink("/kaggle/working/sub/prediction_din_org.zip")

   ID   Task1&2
0   0  0.122313
1   1  0.217005
2   2  0.272411
count    343025.000000
mean          0.199294
std           0.137160
min           0.000494
25%           0.092360
50%           0.163312
75%           0.279140
max           0.733356
Name: Task1&2, dtype: float64


/kaggle/working/sub/prediction_din_org.zip

In [12]:
import pyarrow.parquet as pq
print(pq.ParquetFile(D + "item_raw_feature.parquet").schema)

required group field_id=-1 root {
  optional int64 field_id=-1 item_id;
  optional binary field_id=-1 item_title (String);
  optional group field_id=-1 item_tags (List) {
    repeated group field_id=-1 list {
      optional int64 field_id=-1 element;
    }
  }
  optional int64 field_id=-1 likes_level;
  optional int64 field_id=-1 views_level;
  optional group field_id=-1 txt_emb_BERT (List) {
    repeated group field_id=-1 list {
      optional float field_id=-1 element;
    }
  }
  optional group field_id=-1 img_emb_CLIPRN50 (List) {
    repeated group field_id=-1 list {
      optional float field_id=-1 element;
    }
  }
  optional binary field_id=-1 item_title_cn (String);
}



In [13]:
from sklearn.decomposition import PCA
raw = pd.read_parquet(D + "item_raw_feature.parquet", columns=["item_id", "txt_emb_BERT", "img_emb_CLIPRN50"])
def l2(X): return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-8)
B = l2(np.stack(raw["txt_emb_BERT"].values).astype(np.float32))
C = l2(np.stack(raw["img_emb_CLIPRN50"].values).astype(np.float32))
P = PCA(n_components=128, random_state=0).fit_transform(np.concatenate([B, C], 1)).astype(np.float32)
P *= (np.linalg.norm(E_org[1:], axis=1).mean() / np.linalg.norm(P, axis=1).mean())

E_own = np.zeros_like(E_org); E_own[raw["item_id"].values] = P; E_own[0] = 0
del raw, B, C; gc.collect()

model_own, auc_own = train(E_own, epochs=4)
print("AUC valid: organisateurs =", auc_org, "| nos embeddings =", auc_own)

epoch 1: loss=0.4528  valid AUC=0.7364  (76s)
epoch 2: loss=0.4236  valid AUC=0.7387  (76s)
epoch 3: loss=0.3772  valid AUC=0.7111  (76s)
epoch 4: loss=0.3019  valid AUC=0.6751  (76s)
AUC valid: organisateurs = 0.7372290426596665 | nos embeddings = 0.73867300360629


In [ ]:
def train2(E, epochs=3, bs=2048, lr=1e-3, emb_lr=1e-4, wd=1e-4, drop=0.3, freeze_emb=False, seed=0):
    torch.manual_seed(seed)
    m = DIN(E, NL, NV, drop=drop).to(dev)
    m.emb.weight.requires_grad = not freeze_emb
    other = [p for n, p in m.named_parameters() if n != "emb.weight"]
    groups = [{"params": other, "lr": lr, "weight_decay": wd}]
    if not freeze_emb:
        groups.append({"params": [m.emb.weight], "lr": emb_lr, "weight_decay": 0.0})
    opt = torch.optim.AdamW(groups)
    n = len(TR["t"]); steps = epochs * ((n + bs - 1) // bs)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: max(0.05, 1 - s / steps))
    scaler = torch.amp.GradScaler(enabled=dev.type == "cuda")
    best_auc, best_state = 0, None
    for ep in range(epochs):
        m.train(); perm = torch.randperm(n, device=dev); t0 = time.time(); tot = torch.zeros((), device=dev)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            loss = F.binary_cross_entropy_with_logits(fwd(m, TR, idx).float(), TR["y"][idx])
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.detach()
        auc = valid_auc(m)
        print(f"epoch {ep+1}: loss={tot.item()/(n/bs):.4f}  valid AUC={auc:.4f}  ({time.time()-t0:.0f}s)")
        if auc > best_auc:
            best_auc = auc; best_state = {k: v.detach().clone() for k, v in m.state_dict().items()}
    m.load_state_dict(best_state)
    return m, best_auc

res = {}
for name, cfg in [("gelés", dict(freeze_emb=True)), ("emb_lr=1e-4", dict(freeze_emb=False, emb_lr=1e-4))]:
    print("=== embeddings", name, "===")
    res[name] = train2(E_org, **cfg)
print({k: v[1] for k, v in res.items()})

=== embeddings gelés ===
epoch 1: loss=0.4821  valid AUC=0.6668  (34s)
epoch 2: loss=0.4713  valid AUC=0.6845  (34s)
epoch 3: loss=0.4658  valid AUC=0.6907  (34s)
=== embeddings emb_lr=1e-4 ===
epoch 1: loss=0.4643  valid AUC=0.7275  (76s)
epoch 2: loss=0.4407  valid AUC=0.7356  (76s)
epoch 3: loss=0.4335  valid AUC=0.7377  (76s)
{'gelés': np.float64(0.6907002460061036), 'emb_lr=1e-4': np.float64(0.737674299087763)}


In [26]:
def make_submission(m, tag):
    p = predict(m, TE)
    sub = pd.DataFrame({"ID": np.arange(len(p)), "Task1&2": p})
    assert len(sub) == 343025 and sub["Task1&2"].between(0, 1).all() and not sub.isna().any().any()
    os.makedirs("/kaggle/working/sub", exist_ok=True)
    sub.to_csv("/kaggle/working/sub/prediction.csv", index=False)
    with zipfile.ZipFile(f"/kaggle/working/sub/{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("/kaggle/working/sub/prediction.csv", arcname="prediction.csv")
    print(sub.head(3)); print(sub["Task1&2"].describe())

make_submission(res["emb_lr=1e-4"][0], "prediction_din_lr")
from IPython.display import FileLink
FileLink("/kaggle/working/sub/prediction_din_lr.zip")

   ID   Task1&2
0   0  0.083591
1   1  0.167083
2   2  0.227298
count    343025.000000
mean          0.197130
std           0.144476
min           0.000234
25%           0.087564
50%           0.158826
75%           0.268749
max           0.699254
Name: Task1&2, dtype: float64


/kaggle/working/sub/prediction_din_lr.zip

In [27]:
res2 = {}
for name, E in [("org", E_org), ("own", E_own)]:
    print("=== embeddings", name, "===")
    res2[name] = train2(E, epochs=5, freeze_emb=False, emb_lr=1e-4)
print({k: v[1] for k, v in res2.items()})

=== embeddings org ===
epoch 1: loss=0.4640  valid AUC=0.7281  (76s)
epoch 2: loss=0.4405  valid AUC=0.7360  (76s)
epoch 3: loss=0.4332  valid AUC=0.7385  (76s)
epoch 4: loss=0.4282  valid AUC=0.7388  (76s)
epoch 5: loss=0.4243  valid AUC=0.7387  (75s)
=== embeddings own ===
epoch 1: loss=0.4670  valid AUC=0.7230  (76s)
epoch 2: loss=0.4433  valid AUC=0.7331  (76s)
epoch 3: loss=0.4358  valid AUC=0.7362  (76s)
epoch 4: loss=0.4310  valid AUC=0.7368  (76s)
epoch 5: loss=0.4273  valid AUC=0.7365  (76s)
{'org': np.float64(0.7387606626468596), 'own': np.float64(0.7368374539431863)}


In [28]:
def make_submission(m, tag):
    p = predict(m, TE)
    sub = pd.DataFrame({"ID": np.arange(len(p)), "Task1&2": p})
    assert len(sub) == 343025 and sub["Task1&2"].between(0, 1).all() and not sub.isna().any().any()
    os.makedirs("/kaggle/working/sub", exist_ok=True)
    sub.to_csv("/kaggle/working/sub/prediction.csv", index=False)
    with zipfile.ZipFile(f"/kaggle/working/sub/{tag}.zip", "w", zipfile.ZIP_DEFLATED) as z:
        z.write("/kaggle/working/sub/prediction.csv", arcname="prediction.csv")
    print(sub.head(3)); print(sub["Task1&2"].describe())

best = res2["org"][0]
print("AUC valid recalculée:", valid_auc(best))
make_submission(best, "prediction_din_org5")
torch.save(best.state_dict(), "/kaggle/working/model_din_org5.pt")
from IPython.display import FileLink
FileLink("/kaggle/working/sub/prediction_din_org5.zip")

AUC valid recalculée: 0.7387606626468596
   ID   Task1&2
0   0  0.089773
1   1  0.182134
2   2  0.234582
count    343025.000000
mean          0.196428
std           0.149001
min           0.000177
25%           0.084644
50%           0.156749
75%           0.267791
max           0.792464
Name: Task1&2, dtype: float64


/kaggle/working/sub/prediction_din_org5.zip